# WTI Crude Oil Price Forecasting — Stateless Methods: Systematic Backtest (Notebook 4 of 7)

This notebook simulates a rigorous production forecasting workflow:

1. Run a **rolling weekly backtest across 2025** using
   `energy_oil_backtest.yaml` for all candidate predictors.
2. Compute metrics — **CRPS** for 5/10/21-day trajectories.
3. Select the **top contender configurations** based solely on 2025
   historical performance (no peeking at 2026).
4. Let the contenders compete in the **2026 Protected Arena**
   (`energy_oil_eval.yaml`) during the geopolitical price shock —
   measuring adaptive real-time responsiveness and calibration.

All predictors use the same `Predictor` interface introduced in Notebooks 1–2.
Agent configs are imported from `energy_oil_forecasting.analyst_agent`.

---
## 1. Setup, Data Registration & Spec Loading

In [16]:
import warnings
from pathlib import Path

import energy_oil_forecasting
import pandas as pd
import yaml
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
    describe_spec,
)
from energy_oil_forecasting.data import build_wti_service


warnings.filterwarnings("ignore")

# ── Mode ──────────────────────────────────────────────────────────────────────
# Set SMOKE_TEST = True to run a 2-origin, 1-sample version of the notebook
# for fast local development and end-to-end CI testing. The full specs run
# 51 backtest + 8 eval origins; smoke runs 2 + 2.
SMOKE_TEST = False

# ── Model selection ───────────────────────────────────────────────────────────
# Change these two lines to swap models for the whole notebook.
# ADK agents use the bare Gemini model name; litellm (LLMP) needs the
# "gemini/" prefix.
AGENT_MODEL = "gemini-3.1-flash-lite-preview"
LLMP_MODEL = "gemini-3.1-flash-lite-preview"

# ── Derived settings (do not edit below) ─────────────────────────────────────
N_SAMPLES = 1 if SMOKE_TEST else 3  # trajectories per LLMP call

data_service = build_wti_service()

spec_dir = Path(energy_oil_forecasting.__file__).parent / "specs"
if SMOKE_TEST:
    backtest_file, eval_file = "energy_oil_smoke.yaml", "energy_oil_eval_smoke.yaml"
else:
    backtest_file, eval_file = "energy_oil_backtest.yaml", "energy_oil_eval.yaml"

with open(spec_dir / backtest_file) as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))
with open(spec_dir / eval_file) as f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

print(
    f"{'⚡ SMOKE MODE' if SMOKE_TEST else '📊 FULL MODE'} — AGENT_MODEL={AGENT_MODEL!r}  LLMP_MODEL={LLMP_MODEL!r}  N_SAMPLES={N_SAMPLES}"
)
print()
print("━" * 72)
print("LOADED SPECIFICATIONS:")
print("━" * 72)
print(describe_spec(backtest_spec, data_service))
print(describe_spec(eval_spec, data_service))

📊 FULL MODE — AGENT_MODEL='gemini-3.1-flash-lite-preview'  LLMP_MODEL='gemini-3.1-flash-lite-preview'  N_SAMPLES=3

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LOADED SPECIFICATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MultiTargetBacktestSpec (spec_id=energy_oil_backtest)
  description: Weekly rolling backtest in 2025 for daily WTI crude oil price forecasting. Evaluates trajectory forecasts (5, 10, 21 business days) with CRPS/MAE and binary up-shock forecasts (climb > $5 in 5 business days) with Brier Score. Used to select the top contender models.
  start:       2025-01-06 00:00:00
  end:         2025-12-22 00:00:00
  stride:      5
  warmup:      250
  tasks:       1

Task: wti_oil_price_forecast
  description: WTI Crude Oil continuous front-month futures Close price (yfinance symbol: CL=F), projected 5, 10, and 21 trading days ahead.
  horizons:    [5, 10, 21] (len=3)
  frequency:   B
  resolution:  observed_value_at_

---
## 2. Candidate Predictors

The cell below defines `PREDICTORS` — an ordered dict mapping display name to
predictor object.  **Comment out any line to exclude that predictor from the
entire notebook**: backtest, leaderboard, eval, and save cells all loop over
`PREDICTORS` automatically.

| Predictor | Elicitation strategy | Tools |
|---|---|---|
| `LastValuePredictor` | Statistical baseline (carry-forward) | — |
| `ProphetPredictor` | Trend + seasonality decomposition | — |
| `SampledTrajectoryLLMPredictor` | Sample N trajectories → empirical quantiles | — |
| `QuantileGridLLMPredictor` | Direct one-shot quantile elicitation | — |
| `AgentPredictor` (news) | News-grounded agent with search + code execution | Google Search, E2B |

In [17]:
from aieng.forecasting.methods import (
    LastValuePredictor,
    QuantileGridLLMPredictor,
    QuantileGridLLMPredictorConfig,
    SampledTrajectoryLLMPredictor,
    SampledTrajectoryLLMPredictorConfig,
)
from energy_oil_forecasting.analyst_agent import (
    build_wti_agent_predictor,
    build_wti_news_config,
)
from energy_oil_forecasting.prophet_baseline import ProphetPredictor


# ── Active predictors ─────────────────────────────────────────────────────────
# Comment out any line to exclude that predictor from the full notebook run.
# The display name (dict key) is used in leaderboard, eval, and save files.
PREDICTORS = {
    "Naive (Last Value)": LastValuePredictor(),
    "Prophet": ProphetPredictor(predictor_id="prophet_daily"),
    f"LLMP-Sampled ({LLMP_MODEL})": SampledTrajectoryLLMPredictor(
        SampledTrajectoryLLMPredictorConfig(model=LLMP_MODEL, n_samples=N_SAMPLES)
    ),
    f"LLMP-Grid ({LLMP_MODEL})": QuantileGridLLMPredictor(QuantileGridLLMPredictorConfig(model=LLMP_MODEL)),
    f"News Agent ({AGENT_MODEL})": build_wti_agent_predictor(build_wti_news_config(model=AGENT_MODEL)),
}

print(f"Active predictors ({len(PREDICTORS)}):")
for name in PREDICTORS:
    print(f"  {name}")

Candidate predictors:
  last_value_naive
  prophet_daily
  llmp_sampled_trajectories[gemini-3.1-flash-lite-preview]
  llmp_quantile_grid[gemini-3.1-flash-lite-preview]
  agent_predictor_wti_analyst_news_continuous


---
## 3. Run the 2025 Historical Backtest

All 51 weekly origins in 2025 are evaluated for each predictor.
`cached_multi_backtest` caches results under `data/predictions/` so
subsequent runs are instant.

In [18]:
print(f"Running 2025 rolling backtest ({len(PREDICTORS)} predictor(s))...")
print("LLM/agent runs are expensive — first run will take several minutes.\n")

backtest_results: dict[str, object] = {}
for _name, _predictor in PREDICTORS.items():
    backtest_results[_name] = cached_multi_backtest(_predictor, backtest_spec, data_service)
    print(f"  {_name} ✓")

print("\nAll 2025 backtests complete.")

Running 2025 rolling backtest (51 weekly origins × 5 predictors)...
LLM/agent runs are expensive — first run will take several minutes.

LastValue ✓
Prophet ✓


KeyboardInterrupt: 

---
## 4. Compute Metrics and Select Contenders

We score each predictor on:
- **CRPS** (Continuous Ranked Probability Score) across the 5/10/21-day trajectory
- **MAE** at the 21-day horizon (point forecast accuracy)

The top 3 scorers (by mean CRPS) are selected as contenders for the
2026 protected arena. Selection is based solely on 2025 performance.

In [ ]:
from energy_oil_forecasting.analysis import score_backtest_results


all_results = list(backtest_results.items())

leaderboard_rows = []
for name, results in all_results:
    scores = score_backtest_results(results, data_service)
    leaderboard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS": scores.get("mean_crps", float("nan")),
            "MAE h=21d": scores.get("mae_h21", float("nan")),
        }
    )

df_leaderboard = pd.DataFrame(leaderboard_rows).set_index("Predictor")
df_leaderboard = df_leaderboard.sort_values("Mean CRPS")

print("━" * 72)
print("2025 HISTORICAL BACKTEST LEADERBOARD:")
print("━" * 72)
print(df_leaderboard.to_string())

# Select top contenders for eval — at most 3, but never more than available.
N_CONTENDERS = min(3, len(df_leaderboard))
top3 = df_leaderboard.head(N_CONTENDERS).index.tolist()
print(f"\nSelected {N_CONTENDERS} contender(s) for 2026 arena: {top3}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2025 HISTORICAL BACKTEST LEADERBOARD:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                              Mean CRPS  MAE h=21d
Predictor                                                         
LLMP-Grid (gemini-3.1-flash-lite-preview)      4.095949        NaN
LLMP-Sampled (gemini-3.1-flash-lite-preview)   5.074999        NaN
Naive (Last Value)                             5.834998        NaN
News Agent (gemini-3.1-flash-lite-preview)     6.134875        NaN
Prophet                                       12.845009        NaN

Selected contenders for 2026 arena: ['LLMP-Grid (gemini-3.1-flash-lite-preview)', 'LLMP-Sampled (gemini-3.1-flash-lite-preview)', 'Naive (Last Value)']


In [10]:
# ── Save backtest results for NB05 / NB06 ────────────────────────────────────
# Downstream notebooks load these with BacktestResult.model_validate_json().
# The files are gitignored (derived data) and regenerated by re-running NB04.
# cached_multi_backtest returns {task_id: BacktestResult}; extract single task.
_CURRICULUM_DIR = Path("adaptive_agent/curriculum")
_CURRICULUM_DIR.mkdir(exist_ok=True)
for _name, _result_dict in backtest_results.items():
    _result = next(iter(_result_dict.values()))
    (_CURRICULUM_DIR / f"backtest_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
print(f"Saved {len(backtest_results)} backtest result(s) to {_CURRICULUM_DIR}/")

Saved 1 backtest result(s) to adaptive_agent/curriculum/


---
## 5. The 2026 Protected Arena Competition

We evaluate the selected contenders on **8 weekly origins in early 2026**
(`energy_oil_eval.yaml`) — a period of major geopolitical volatility as
Persian Gulf shipping-lane closures drove WTI from ~$71 to above $100.

This is a **prospective evaluation**: the 2026 data was not seen during
contender selection. News-grounded agents retrieve information with a strict
temporal cutoff at each origin, approximating a genuine live-test environment.

In [11]:
print("Running 2026 protected arena evaluation...")
eval_results: dict[str, object] = {}
for name in top3:
    eval_results[name] = cached_multi_backtest(PREDICTORS[name], eval_spec, data_service)
    print(f"  {name} ✓")

print("\n2026 evaluation complete.")

Running 2026 protected arena evaluation...


06:35:12 - cmdstanpy - INFO - Chain [1] start processing
06:35:13 - cmdstanpy - INFO - Chain [1] done processing
06:35:13 - cmdstanpy - INFO - Chain [1] start processing
06:35:14 - cmdstanpy - INFO - Chain [1] done processing


  Prophet ✓

2026 evaluation complete.


In [ ]:
# ── Save eval results for NB06 ───────────────────────────────────────────────
# eval_results is {predictor_name: {task_id: BacktestResult}}; extract single task.
for _name, _result_dict in eval_results.items():
    _result = next(iter(_result_dict.values()))
    (_CURRICULUM_DIR / f"eval_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
print(f"Saved {len(eval_results)} eval result(s) to {_CURRICULUM_DIR}/")

---
## 6. Visualisation & Scorecard

We compare how each contender reacted as the price shock unfolded.
Statistical models like Prophet expect mean-reversion and miss the breakout.
The news-grounded agent reads real-time intelligence and adjusts its forecast
accordingly — at the cost of higher compute and latency.

In [ ]:
from energy_oil_forecasting.analysis import score_backtest_results


scorecard_rows = []
for name in top3:
    scores = score_backtest_results(eval_results[name], data_service)
    scorecard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS (2026)": scores.get("mean_crps", float("nan")),
            "MAE h=21d (2026)": scores.get("mae_h21", float("nan")),
            "80% CI Coverage": scores.get("coverage_80", float("nan")),
        }
    )

df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor")
df_scorecard = df_scorecard.sort_values("Mean CRPS (2026)")

print("━" * 72)
print("FINAL 2026 PROTECTED ARENA SCORECARD:")
print("━" * 72)
print(df_scorecard.to_string())

---
## 7. Core Takeaways

1. **Statistical models** (Prophet, Last Value) are strong in stable regimes.
   During structural price shocks they extrapolate past trends, missing the
   breakout and producing catastrophically narrow intervals.

2. **Direct-prompt LLMPs** have an implicit knowledge cutoff. For 2026
   origins they may have partial training signal about early 2026 events,
   but cannot access post-cutoff news in real time. The two LLMP variants
   (`SampledTrajectoryLLMPredictor` vs. `QuantileGridLLMPredictor`) differ
   in elicitation strategy and cost: sampled trajectories give an empirical
   distribution over N calls; quantile-grid elicitation asks the model to
   output the full quantile grid in one shot. Calibration differences between
   them are an open empirical question worth examining here.

3. **News-grounded agents** with bounded search incorporate real-time
   market intelligence, enabling a much faster response to structural
   shocks — at higher compute cost and non-zero leakage risk through
   the search tool.

4. **The `Predictor` abstraction makes all of this composable.** The same
   backtest harness, scoring functions, and visualisation tools work
   equally for Prophet, LLMP, and agent predictors.

---
## 8. What stateless methods can't do

Every method evaluated in this notebook is **stateless** — each prediction is
made fresh, with no memory of past errors or resolved outcomes.  This is the
right setting for an apples-to-apples comparison, but it leaves a systematic
gap:

- **No error feedback.** If a method consistently produces intervals that are
  too narrow during high-volatility regimes, it will keep making the same
  mistake.  There is no mechanism to update its calibration between rounds.

- **No strategy evolution.** A news-grounded agent starts with the same
  prior on every call.  It cannot accumulate experience about which signals
  it over- or under-weights, or refine its approach as the market evolves.

- **No learning from resolution.** When a forecast resolves, the result
  disappears.  A human analyst would use the error to interrogate their
  process.  A stateless model cannot.

→ **Notebook 5** introduces an adaptive agent that keeps a mutable strategy
skill, studies 2024 historical performance as training material, and is
evaluated on held-out 2025+ data — the same origins used here — so you can
measure what learning adds.